<a href="https://colab.research.google.com/github/SharvChopra/LLM_Code/blob/main/Tokenizer_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
## character level tokenization

class Tokenizer:
  def encode(self,text):
    return [ord(c) for c in text]

  def decode(self,token):
    return "".join(chr(t) for t in token)

token = Tokenizer()


my_text = "Hello, AI!"
print(f"Original Text: '{my_text}'")

# 3. Encode the text (Human -> Machine)
encoded_tokens = token.encode(my_text)
print(f"Encoded Tokens: {encoded_tokens}")

# 4. Decode the tokens (Machine -> Human)
decoded_text = token.decode(encoded_tokens)
print(f"Decoded Text:  '{decoded_text}'")

Original Text: 'Hello, AI!'
Encoded Tokens: [72, 101, 108, 108, 111, 44, 32, 65, 73, 33]
Decoded Text:  'Hello, AI!'


In [4]:
from collections import Counter

class BPE_Tokenizer:
  def __init__(self):
    ## 2 empty dictionaries
    self.merges = {}
    self.vocab = {}

  def _get_pairs(self,tokens):
    pairs = Counter()
    for i in range(len(tokens)-1):
      pairs[(tokens[i], tokens[i + 1])] += 1
    return pairs

  def _merge_pair(self,tokens,pair,new_token):
    merged = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
          merged.append(new_token)
          i += 2
        else:
          merged.append(tokens[i])
          i += 1
    return merged

  def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        self.vocab = {i: bytes([i]) for i in range(256)}

        for i in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            new_token = 256 + i
            tokens = self._merge_pair(tokens, best_pair, new_token)
            self.merges[best_pair] = new_token
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

        return self

  def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)
        return tokens

  def decode(self, tokens):
      byte_sequence = b"".join(self.vocab[t] for t in tokens)
      return byte_sequence.decode("utf-8", errors="replace")

In [5]:
corpus = (
    "The cat sat on the mat. The cat ate the rat. "
    "The dog sat on the log. The dog ate the frog. "
    "Natural language processing is the study of how computers "
    "understand and generate human language. "
    "Tokenization is the first step in any NLP pipeline."
)

tokenizer = BPE_Tokenizer()
tokenizer.train(corpus, num_merges=40)

test_sentences = [
    "The cat sat on the mat.",
    "Natural language processing",
    "tokenization pipeline",
    "unhappiness",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded)
    raw_bytes = len(sentence.encode("utf-8"))
    ratio = len(encoded) / raw_bytes
    print(f"'{sentence}'")
    print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes) -- ratio: {ratio:.2f}")
    print(f"  Roundtrip: {'PASS' if decoded == sentence else 'FAIL'}")

'The cat sat on the mat.'
  Tokens: 3 (from 23 bytes) -- ratio: 0.13
  Roundtrip: PASS
'Natural language processing'
  Tokens: 17 (from 27 bytes) -- ratio: 0.63
  Roundtrip: PASS
'tokenization pipeline'
  Tokens: 16 (from 21 bytes) -- ratio: 0.76
  Roundtrip: PASS
'unhappiness'
  Tokens: 10 (from 11 bytes) -- ratio: 0.91
  Roundtrip: PASS


In [6]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = [
    "The cat sat on the mat.",
    "unhappiness",
    "Hello, world!",
    "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
    "Geschwindigkeitsbegrenzung",
]

for text in texts:
    our_tokens = tokenizer.encode(text)
    tiktoken_tokens = enc.encode(text)
    tiktoken_pieces = [enc.decode([t]) for t in tiktoken_tokens]
    print(f"'{text}'")
    print(f"  Our BPE:   {len(our_tokens)} tokens")
    print(f"  tiktoken:  {len(tiktoken_tokens)} tokens -> {tiktoken_pieces}")

'The cat sat on the mat.'
  Our BPE:   3 tokens
  tiktoken:  7 tokens -> ['The', ' cat', ' sat', ' on', ' the', ' mat', '.']
'unhappiness'
  Our BPE:   10 tokens
  tiktoken:  3 tokens -> ['un', 'h', 'appiness']
'Hello, world!'
  Our BPE:   13 tokens
  tiktoken:  4 tokens -> ['Hello', ',', ' world', '!']
'def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)'
  Our BPE:   68 tokens
  tiktoken:  23 tokens -> ['def', ' fibonacci', '(n', '):', ' return', ' n', ' if', ' n', ' <', ' ', '2', ' else', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']
'Geschwindigkeitsbegrenzung'
  Our BPE:   24 tokens
  tiktoken:  9 tokens -> ['G', 'esch', 'wind', 'ig', 'ke', 'its', 'beg', 'ren', 'zung']


In [7]:
def analyze_vocabulary(tokenizer, test_texts):
    total_tokens = 0
    total_chars = 0
    token_usage = Counter()

    for text in test_texts:
        encoded = tokenizer.encode(text)
        total_tokens += len(encoded)
        total_chars += len(text)
        for t in encoded:
            token_usage[t] += 1

    print(f"Vocabulary size: {len(tokenizer.vocab)}")
    print(f"Total tokens across all texts: {total_tokens}")
    print(f"Total characters: {total_chars}")
    print(f"Avg tokens per character: {total_tokens / total_chars:.2f}")

    print(f"\nMost used tokens:")
    for token_id, count in token_usage.most_common(10):
        token_bytes = tokenizer.vocab[token_id]
        display = token_bytes.decode("utf-8", errors="replace")
        print(f"  Token {token_id:4d}: '{display}' (used {count} times)")

    unused = [t for t in tokenizer.vocab if t not in token_usage]
    print(f"\nUnused tokens: {len(unused)} out of {len(tokenizer.vocab)}")

Tiktoken - used by OpenAI

In [8]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

text = "Tokenizers convert text into numbers"
tokens = enc.encode(text)
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

decoded_text = enc.decode(tokens)
print(f"Decoded text: {decoded_text}")

Tokens: [3404, 12509, 5625, 1495, 1139, 5219]
Number of tokens: 6
Decoded text: Tokenizers convert text into numbers


In [14]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(vocab_size=1000, special_tokens=["<pad>", "<eos>", "<unk>"])


output = tokenizer.encode("The cat sat on the mat.")
print(f"Tokens: {output.tokens}")
print(f"IDs: {output.ids}")

Tokens: []
IDs: []


# **PROUCTION READY BPE ALGORITHM**

In [15]:
def bytes_to_token(text):
  return list(text.encode("utf-8"))

def tokens_to_text(token_bytes):
    return bytes(token_bytes).decode("utf-8", errors="replace")

In [16]:
## test on multilingual languages
texts = [
    ("English", "hello"),
    ("Chinese", "你好"),
    ("Emoji", "🔥"),
    ("Mixed", "hello你好🔥"),
]

for label, text in texts:
    b = bytes_to_token(text)
    print(f"{label}: {len(text)} chars -> {len(b)} bytes -> {b}")

English: 5 chars -> 5 bytes -> [104, 101, 108, 108, 111]
Chinese: 2 chars -> 6 bytes -> [228, 189, 160, 229, 165, 189]
Emoji: 1 chars -> 4 bytes -> [240, 159, 148, 165]
Mixed: 8 chars -> 15 bytes -> [104, 101, 108, 108, 111, 228, 189, 160, 229, 165, 189, 240, 159, 148, 165]


Stage - 2

In [17]:
import re

try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )

def pre_tokenize(text):
    return [match.group() for match in GPT2_PATTERN.finditer(text)]

In [18]:
print(pre_tokenize("Hello, world! Don't stop."))
# [' Hello', ',', ' world', '!', " Don", "'t", ' stop', '.']

['Hello', ',', ' world', '!', ' Don', "'t", ' stop', '.']


Stage - 3

In [20]:
## BPE-Level Algorihtm
from collections import Counter

def get_byte_pairs(chunks):
    pairs = Counter()
    for chunk in chunks:
        byte_seq = list(chunk.encode("utf-8"))
        for i in range(len(byte_seq) - 1):
            pairs[(byte_seq[i], byte_seq[i + 1])] += 1
    return pairs

def apply_merge(byte_seq, pair, new_id):
    merged = []
    i = 0
    while i < len(byte_seq):
        if i < len(byte_seq) - 1 and byte_seq[i] == pair[0] and byte_seq[i + 1] == pair[1]:
            merged.append(new_id)
            i += 2
        else:
            merged.append(byte_seq[i])
            i += 1
    return merged

Stage - 4 Special Token Handling

In [21]:
class SpecialTokenHandler:
    def __init__(self):
        self.special_tokens = {}
        self.pattern = None

    def add_token(self, token_str, token_id):
        self.special_tokens[token_str] = token_id
        escaped = [re.escape(t) for t in sorted(self.special_tokens.keys(), key=len, reverse=True)]
        self.pattern = re.compile("|".join(escaped))

    def split_with_specials(self, text):
        if not self.pattern:
            return [(text, False)]
        parts = []
        last_end = 0
        for match in self.pattern.finditer(text):
            if match.start() > last_end:
                parts.append((text[last_end:match.start()], False))
            parts.append((match.group(), True))
            last_end = match.end()
        if last_end < len(text):
            parts.append((text[last_end:], False))
        return parts

Stage - 5 Full tokenizer class

In [22]:
import unicodedata

class ProductionTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.special_handler = SpecialTokenHandler()
        self.next_id = 256

    def normalize(self, text):
        return unicodedata.normalize("NFKC", text)

    def train(self, text, num_merges):
        text = self.normalize(text)
        chunks = pre_tokenize(text)
        chunk_bytes = [list(chunk.encode("utf-8")) for chunk in chunks]

        for i in range(num_merges):
            pairs = Counter()
            for seq in chunk_bytes:
                for j in range(len(seq) - 1):
                    pairs[(seq[j], seq[j + 1])] += 1
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            new_id = self.next_id
            self.next_id += 1
            self.merges[best] = new_id
            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]
            chunk_bytes = [apply_merge(seq, best, new_id) for seq in chunk_bytes]

    def add_special_token(self, token_str):
        token_id = self.next_id
        self.next_id += 1
        self.special_handler.add_token(token_str, token_id)
        self.vocab[token_id] = token_str.encode("utf-8")
        return token_id

    def encode(self, text):
        text = self.normalize(text)
        parts = self.special_handler.split_with_specials(text)
        all_ids = []
        for part_text, is_special in parts:
            if is_special:
                all_ids.append(self.special_handler.special_tokens[part_text])
            else:
                for chunk in pre_tokenize(part_text):
                    byte_seq = list(chunk.encode("utf-8"))
                    for pair, new_id in self.merges.items():
                        byte_seq = apply_merge(byte_seq, pair, new_id)
                    all_ids.extend(byte_seq)
        return all_ids

    def decode(self, ids):
        byte_parts = []
        for token_id in ids:
            if token_id in self.vocab:
                byte_parts.append(self.vocab[token_id])
        return b"".join(byte_parts).decode("utf-8", errors="replace")

    def vocab_size(self):
        return len(self.vocab)

In [23]:
corpus = (
    "The quick brown fox jumps over the lazy dog. "
    "The quick brown fox runs through the forest. "
    "Machine learning models process natural language. "
    "Deep learning transforms how we build software. "
    "def train(model, data): return model.fit(data) "
    "def predict(model, x): return model(x) "
)

tok = ProductionTokenizer()
tok.train(corpus, num_merges=50)

bos = tok.add_special_token("<|begin|>")
eos = tok.add_special_token("<|end|>")

test_texts = [
    "The quick brown fox.",
    "你好世界",
    "Hello 🌍 World",
    "def foo(x): return x + 1",
    f"<|begin|>Hello<|end|>",
]

for text in test_texts:
    ids = tok.encode(text)
    decoded = tok.decode(ids)
    print(f"Input:   {text}")
    print(f"Tokens:  {len(ids)} ids")
    print(f"Decoded: {decoded}")
    print()

Input:   The quick brown fox.
Tokens:  5 ids
Decoded: The quick brown fox.

Input:   你好世界
Tokens:  12 ids
Decoded: 你好世界

Input:   Hello 🌍 World
Tokens:  16 ids
Decoded: Hello 🌍 World

Input:   def foo(x): return x + 1
Tokens:  14 ids
Decoded: def foo(x): return x + 1

Input:   <|begin|>Hello<|end|>
Tokens:  7 ids
Decoded: <|begin|>Hello<|end|>

